# 📦 Build Training Data Zip

**Run once**, then upload zip to Google Drive. Colab training fetches from Drive.

Data sources: ESC-50 + UrbanSound8K + Xeno-Canto

In [ ]:
# @title 1. Setup
!pip install -q pydub torchaudio
!apt-get install -q ffmpeg

import os, sys, json, time, shutil, csv, tarfile, urllib.request, urllib.parse
from pydub import AudioSegment

DATA = "data/animal1000"
os.makedirs(DATA, exist_ok=True)
for cls in ['Dog','Cat','Rooster','Frog','Crow','Insect','Hen']:
    os.makedirs(f"{DATA}/{cls}", exist_ok=True)
print("✅ Directories ready")

In [ ]:
# @title 2. ESC-50 (640 files)
!wget -q https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip -O /tmp/esc50.zip
!unzip -qo /tmp/esc50.zip -d /tmp/
!python -c "
import os, shutil
cls_map = {
    'dog': 'Dog', 'cat': 'Cat', 'rooster': 'Rooster',
    'frog': 'Frog', 'crow': 'Crow', 'insect': 'Insect', 'cricket': 'Insect',
    'hen': 'Hen', 'chicken': 'Hen', 'chirping_birds': 'Crow'
}
import csv, shutil
audio_dir = '/tmp/ESC-50-master/audio'
with open('/tmp/ESC-50-master/meta/esc50.csv') as f:
    for row in csv.DictReader(f):
        cat = row['category'].lower()
        for kw, cls in cls_map.items():
            if kw in cat:
                src = f'{audio_dir}/{row[\"filename\"]}'
                dst = f'data/animal1000/{cls}/{row[\"filename\"]}'
                if os.path.exists(src):
                    shutil.copy2(src, dst)
                break
"
for cls in ['Dog','Cat','Rooster','Frog','Crow','Insect','Hen']:
    n = len(os.listdir(f'data/animal1000/{cls}'))
    print(f'  {cls}: {n}')

In [ ]:
# @title 3. UrbanSound8K → Dog barks (~1,000)
import tarfile, shutil
if not os.path.exists('/tmp/UrbanSound8K'):
    !wget -q https://zenodo.org/records/1203745/files/UrbanSound8K.tar.gz -O /tmp/us8k.tar.gz
    print('Extracting...')
    with tarfile.open('/tmp/us8k.tar.gz') as tar:
        tar.extractall(path='/tmp/')

with open('/tmp/UrbanSound8K/metadata/UrbanSound8K.csv') as f:
    for row in csv.DictReader(f):
        if 'dog' in row['class'].lower():
            src = f"/tmp/UrbanSound8K/audio/fold{row['fold']}/{row['slice_file_name']}"
            dst = f"data/animal1000/Dog/d_{row['slice_file_name']}"
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.copy2(src, dst)

n = len(os.listdir('data/animal1000/Dog'))
print(f'  Dog: {n} files')

In [ ]:
# @title 4. Xeno-Canto → Crow, Rooster, Hen, Frog
QUERIES = {'Crow': 'Corvus', 'Rooster': 'Gallus gallus', 'Hen': 'Gallus gallus', 'Frog': 'frog'}

for cls, query in QUERIES.items():
    print(f'  {cls}: searching...')
    url = f"https://xeno-canto.org/api/2/recordings?query={urllib.parse.quote(query)}"
    try:
        with urllib.request.urlopen(url) as r:
            data = json.loads(r.read())
        recs = data.get('recordings', [])
        print(f'    Found {len(recs)} recordings, downloading up to 500...')
        downloaded = 0
        for rec in recs:
            if downloaded >= 500: break
            url = rec.get('file','')
            if not url.startswith('http'): continue
            path = f"data/animal1000/{cls}/xc_{rec['id']}.mp3"
            if os.path.exists(path):
                downloaded += 1; continue
            try:
                urllib.request.urlretrieve(url, path)
                downloaded += 1
                if downloaded % 100 == 0: print(f'      {downloaded}/500')
                time.sleep(0.3)
            except: pass
        print(f'    Download complete: {downloaded}')
    except Exception as e:
        print(f'    ❌ {e}')

# Convert mp3 to wav
print('\nConverting mp3 → wav...')
for cls in ['Crow','Rooster','Hen','Frog']:
    for f in os.listdir(f'data/animal1000/{cls}'):
        if f.endswith('.mp3'):
            try:
                audio = AudioSegment.from_mp3(f'data/animal1000/{cls}/{f}')
                audio = audio.set_frame_rate(22050).set_channels(1)
                wav_path = f'data/animal1000/{cls}/{f.replace(".mp3",".wav")}'
                audio.export(wav_path, format='wav')
                os.remove(f'data/animal1000/{cls}/{f}')
            except: pass
    n = len(os.listdir(f'data/animal1000/{cls}'))
    print(f'  {cls}: {n} wav files')

In [ ]:
# @title 5. Summary + Zip for Drive
print('📊 Final Count:')
total = 0
for cls in ['Dog','Cat','Rooster','Frog','Crow','Insect','Hen']:
    n = len([f for f in os.listdir(f'data/animal1000/{cls}') if f.endswith('.wav')])
    total += n
    bar = '█'*(n//20)
    print(f'  {cls:<12}: {n:4d} {bar}')
print(f'  Total: {total}')

# Zip
!zip -r /content/animal1000.zip data/animal1000/*.wav -i "*.wav" 2>/dev/null || true
!zip -r /content/animal1000.zip data/animal1000/  2>/dev/null

size = os.path.getsize('/content/animal1000.zip') / 1024**2
print(f'\n📦 /content/animal1000.zip ({size:.0f} MB)')
print('Upload this to: MyDrive/animal_sound_generator/data/')

In [ ]:
# @title 6. Upload to Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/animal_sound_generator/data/
!cp /content/animal1000.zip /content/drive/MyDrive/animal_sound_generator/data/
print('✅ Uploaded to Drive: MyDrive/animal_sound_generator/data/animal1000.zip')